In [32]:
import requests
import pandas as pd

url = "http://127.0.0.1:8000/upload_config_from_csv/1/"

# input_csv = "emd_serono_phi.csv"
# input_csv = "phi_p1.csv"
# input_csv = "phi_p2.csv"
input_csv = "phi_lilly.csv"

# files = {"file": ("all_tables.csv", open(input_csv, "rb"), "text/csv")}

# response = requests.post(url, files=files)

# print(response.status_code, response.json())

In [33]:
df = pd.read_csv(input_csv)
tables = df['TABLE_NAME'].unique()
len(tables)

1

In [34]:
notes_table = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'NOTES']['TABLE_NAME'].to_list()))
len(notes_table)

1

In [35]:
no_notes_df = df[~df['TABLE_NAME'].isin(notes_table)]
no_notes_table = list(set(no_notes_df['TABLE_NAME'].to_list()))
len(no_notes_table)

0

In [36]:
static_df = no_notes_df[no_notes_df['DE_IDENTIFICATION_RULE'] == 'STATIC_OFFSET']
static_tables = list(set(static_df['TABLE_NAME'].to_list()))
len(static_tables)

0

In [37]:
off_tables = list(set(df[df['DE_IDENTIFICATION_RULE'] == 'PATIENT_OFFSET']['TABLE_NAME'].to_list()))
len(off_tables)

0

In [38]:
df['DE_IDENTIFICATION_RULE'].unique()

array([nan, 'PATIENT_ID', 'NOTES'], dtype=object)

In [39]:
import os
import django
import sys
# Set up Django environment
sys.path.append('/Users/neurodiscoveryai/Desktop/deidentification/deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

import pandas as pd
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.views.db_views import run_stats_generation_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [40]:
def fix_reference_value(table_id):
    table = TableDetailsModel.objects.get(id=table_id)
    
    pid_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "PATIENT_ID"
    ]
    enc_col = [
        col['column_name'] for col in table.table_details_for_ui['columns_details'] if col["de_identification_rule"] == "ENCOUNTER_ID"
    ]
    print(pid_col)
    print(enc_col)
    patient_id = pid_col[0] if len(pid_col)>0 else None
    enc_id = enc_col[0] if len(enc_col)>0 else None
    table.table_details_for_ui['reference_enc_id_column'] = enc_id
    table.table_details_for_ui['reference_patient_id_column'] = patient_id
    table.save()

In [41]:
to = TableDetailsModel.objects.get(table_name="edi_inv_cpt")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvoiceId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'edi_inv_cpt',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [42]:
to = TableDetailsModel.objects.get(table_name="edi_inv_diagnosis")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvoiceId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'edi_inv_diagnosis',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [43]:
to = TableDetailsModel.objects.get(table_name="hcfa")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'Id',
   'source_column': 'InvId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'hcfa',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}
to.save()

In [44]:
to = TableDetailsModel.objects.get(table_name="hl7labdatadetail")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
   'source_column': 'ReportId',
   'reference_table': 'labdata'}],
 'source_table': 'hl7labdatadetail',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.save()

In [45]:
to = TableDetailsModel.objects.get(table_name="hl7labnotes")
to.table_details_for_ui['reference_mapping'] = {'conditions': [{'column_name': 'ReportId',
   'source_column': 'reportid',
   'reference_table': 'labdata'}],
 'source_table': 'hl7labnotes',
 'destination_column': 'EncounterId',
 'destination_column_type': 'encounter_id'}
to.save()

In [46]:
to = TableDetailsModel.objects.get(table_name="hcfa")
to.table_details_for_ui['reference_mapping']

{'conditions': [{'column_name': 'Id',
   'source_column': 'InvId',
   'reference_table': 'edi_invoice'}],
 'source_table': 'hcfa',
 'destination_column': 'PatientId',
 'destination_column_type': 'patient_id'}

In [47]:
TableDetailsModel.objects.filter(table_status=0).count()

0

In [48]:
TableDetailsModel.objects.filter(table_status=1).count()

22

In [49]:
TableDetailsModel.objects.filter(table_status=2).count()

1001

In [50]:
TableDetailsModel.objects.filter(table_status=3).count()

0

In [51]:
Task.objects.filter(status=0).count()

0

In [52]:
Task.objects.filter(status=1).count()

0

In [53]:
Task.objects.filter(status=2).count()

1088

In [54]:
Task.objects.filter(status=-1).count()

322

In [55]:
# Task.objects.filter(status=0).update(failure_count=0)
# Task.objects.filter(status=0).update(status=-1)

In [56]:
# Chain.objects.all().delete()

In [57]:
from django.conf import settings
settings.BATCH_SIZE_DURING_DE_IDENTIFICATION, settings.DEFAULT_OFFSET_VALUE

(10000, 34)

In [58]:
# tables = [table_obj.table_name for table_obj in TableDetailsModel.objects.filter(table_status=1)]
# len(tables)

In [59]:
# tables = ['patients']
completed = []
failed_tables = []
len(tables), len(completed), len(failed_tables), len(notes_table)

(1, 0, 0, 1)

In [61]:
totalrun = 0
for table in tables:
    print(table)
    tableobj = TableDetailsModel.objects.get(table_name=table)
    Task.objects.filter(arguments__table_id=tableobj.id).delete()
    # if tableobj.table_status in [2]:
    #     continue
    fix_reference_value(tableobj.id)
    tableobj.refresh_from_db()
    tasks, chain = create_deidentification_task(tableobj, True)
    totalrun += 1
    print(tableobj.rows_count)

2025-12-04 09:49:03,669 - deIdentification.nd_logger - INFO - inside marking as in progress if required


lilly
['patientid']
[]


2025-12-04 09:49:04,243 - deIdentification.nd_logger - INFO - Table lilly dropped from the database.
2025-12-04 09:49:04,245 - deIdentification.nd_logger - INFO - Dropping ignore rows for lilly, deid_prod


638
